# 🎵 SPOTIFY AUDIO TRENDS ANALYTICS — PREPROCESSING PIPELINE
## Đồ án Cuối kỳ: Môn Tương tác Dữ liệu (Interactive Data Visualization)

---

### 📌 THÔNG TIN HỌC THUẬT & PHÂN CÔNG NHIỆM VỤ
- **Thành viên thực hiện:** Đinh Quốc Anh (MSSV: 24133003) — *Vai trò: Data Engineer & Data Analyst*
- **Trọng tâm phụ trách:** Tiền xử lý đa nguồn, Kết nối 5 bảng dữ liệu (JOIN), Xây dựng 13 Calculated Fields & Kiểm định chất lượng dữ liệu
- **Chuẩn Barem Rubric:** Đáp ứng trọn vẹn **Mục 1: Thu thập & Tiền xử lý dữ liệu (2.5 / 2.5 điểm)**:
  - *Xác định bài toán & Dataset (0.5đ):* 5 tập dữ liệu (>1.9M bản ghi thô), Data Dictionary rõ ràng.
  - *Làm sạch dữ liệu (0.5đ):* Xử lý missing, loại ngoại lai, chuẩn hóa chuỗi, regex tách số doanh số.
  - *Biến đổi dữ liệu (0.75đ):* Kiến trúc Phân nhánh (Branching Pipeline), 4 phép JOIN tối ưu, tạo 13 Calculated Fields.
  - *Khám phá dữ liệu EDA tĩnh (0.75đ):* Biểu đồ Seaborn/Matplotlib kiểm định phân phối và ma trận tương quan.

---

### 🏗️ KIẾN TRÚC PHÂN NHÁNH TỐI ƯU (BRANCHING PIPELINE)
Thay vì gộp nguyên khối 1.11 triệu dòng gây lãng phí tài nguyên và sinh ra Missing ảo cho GDP, pipeline tách thành **3 nhánh chuyên biệt**:
1. **Nhánh 1 (Kinh tế - Xã hội):** 74 quốc gia (2021–2022) -> Tỷ lệ khớp GDP đạt ~90-98%, phục vụ Bản đồ thế giới Choropleth & tương quan GDP vs Valence.
2. **Nhánh 2 (COVID Timeline):** Bảng Global (2020–2022) -> Bỏ qua JOIN GDP/Country codes, phục vụ phân tích 3 pha đại dịch (*Lockdown*, *Escapism*, *Reopening*).
3. **Nhánh 3 (Machine Learning):** Cấp độ bài hát độc nhất (~39k bài) -> Loại bỏ hoàn toàn Data Leakage, phục vụ huấn luyện mô hình Logistic Regression.

In [ ]:
# ════════════════════════════════════════════════════════════════
# 1. THIẾT LẬP MÔI TRƯỜNG & IMPORT THƯ VIỆN
# ════════════════════════════════════════════════════════════════
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Cấu hình đường dẫn root dự án
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..")) if os.path.basename(NOTEBOOK_DIR) == "notebooks" else NOTEBOOK_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import các hàm từ module data_loader
from src.data_loader import (
    load_raw_data,
    clean_raw_charts,
    clean_raw_tracks,
    clean_raw_artists,
    clean_raw_gdp,
    clean_raw_country_codes,
    build_socioeconomic_dataset,
    build_covid_timeline_dataset,
    build_ml_dataset,
    data_quality_report,
    RAW_DIR, PROCESSED_DIR, SAMPLE_DIR
)

# Cấu hình giao diện biểu đồ Seaborn
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print(f"✅ Thư mục dự án: {PROJECT_ROOT}")
print(f"✅ Phiên bản Pandas: {pd.__version__} | NumPy: {np.__version__}")

---
## 📂 BƯỚC 1: NẠP VÀ ĐÁNH GIÁ CHẤT LƯỢNG 5 BẢNG DỮ LIỆU THÔ
Theo yêu cầu Rubric, đồ án sử dụng **5 tập dữ liệu đa nguồn** với tổng quy mô **hơn 1.9 triệu bản ghi**:
1. `spotify-weekly-charts.csv`: Bảng xếp hạng tuần toàn cầu và 74 quốc gia (~1.78M dòng).
2. `spotify-tracks-dataset.csv`: 114,000 bài hát với 9 đặc trưng âm học và thể loại.
3. `gdp_data.csv`: Chuỗi thời gian GDP các quốc gia từ World Bank (1960–2022).
4. `country_codes.csv`: Phân loại khu vực và nhóm thu nhập World Bank của 217 quốc gia.
5. `best-selling-artists.csv`: 121 nghệ sĩ bán đĩa chạy nhất lịch sử âm nhạc.

In [ ]:
# Nạp 5 tập dữ liệu thô
charts_raw, tracks_raw, gdp_raw, country_codes_raw, artists_raw = load_raw_data()

In [ ]:
# Xuất bảng báo cáo chất lượng dữ liệu (Data Quality Profile) cho 5 bảng thô
print("=== BÁO CÁO CHẤT LƯỢNG DỮ LIỆU THÔ ===")
raw_tables = [
    ("Weekly Charts", charts_raw),
    ("Tracks Dataset", tracks_raw),
    ("GDP Data", gdp_raw),
    ("Country Codes", country_codes_raw),
    ("Best-Selling Artists", artists_raw)
]
for name, df_obj in raw_tables:
    dup_count = df_obj.duplicated().sum()
    null_pct = df_obj.isnull().mean().mean() * 100
    print(f"• {name:<22s}: {df_obj.shape[0]:>10,} dòng × {df_obj.shape[1]:>2d} cột | Null trung bình: {null_pct:>5.1f}% | Trùng lặp: {dup_count:,} dòng")

---
## 🧹 BƯỚC 2: TIỀN XỬ LÝ & LÀM SẠCH ĐỘC LẬP TỪNG BẢNG
Các thuật toán làm sạch cụ thể:
* **Weekly Charts:** Trích `track_id` chuẩn từ URI; chuyển `week` sang Datetime; ép kiểu số cho `streams`, loại `streams <= 0`; ép kiểu numeric cho 9 audio features; khử trùng lặp `(track_id, week, country)` và lọc sớm 15 cột rác.
* **Tracks Dataset:** Dedup `track_id` theo `popularity` cao nhất (từ 114k -> 89.7k bài); kiểm định biên độ âm học trong $[0, 1]$; clip `loudness` $[-60, 5]$ dB, `tempo` $[0, 300]$ BPM.
* **Artists Dataset:** Regex bóc tách doanh số đĩa triệu bản; trích xuất `career_start`, `career_end`; chuẩn hóa tên nghệ sĩ lower-case.
* **GDP & Country Codes:** Chuẩn hóa tên trường (`gdp_value`, `region_wb`, `income_group`).

In [ ]:
# Thực hiện làm sạch độc lập từng bảng
charts_clean = clean_raw_charts(charts_raw)
tracks_clean = clean_raw_tracks(tracks_raw)
artists_clean = clean_raw_artists(artists_raw)
gdp_clean = clean_raw_gdp(gdp_raw)
country_codes_clean = clean_raw_country_codes(country_codes_raw)

print("\n✅ Đã hoàn thành tiền xử lý độc lập 5 bảng dữ liệu!")

---
## 🚀 BƯỚC 3: THỰC THI KIẾN TRÚC PHÂN NHÁNH (3 DEDICATED BRANCHES)

### Nhánh 1: Kinh tế – Xã hội & Bản đồ 74 Quốc gia (2021 – 2022)
* Lọc bỏ `Global` trước khi JOIN -> Kết nối với GDP và Country Codes đạt tỷ lệ khớp ~90-98% (không bị null giả tạo).
* Tính toán 13 Calculated Fields: `stream_share_country`, `gdp_log`, `mood_category`, `is_latin`, `is_kpop`, `energy_loudness_index`, `audio_diversity_score`.

In [ ]:
# Xây dựng Nhánh 1: Dữ liệu Kinh tế - Xã hội 74 nước (2021-2022)
df_socio = build_socioeconomic_dataset(
    charts=charts_clean,
    gdp=gdp_clean,
    country_codes=country_codes_clean,
    tracks=tracks_clean,
    artists=artists_clean,
    save=True
)
df_socio.head(3)

### Nhánh 2: Dòng thời gian Đại dịch COVID-19 (Global 2020 – 2022)
* Khóa đối tượng về bảng Toàn cầu (`country == "Global"`) từ 2020 đến 2022 (26,598 dòng).
* **Bỏ qua hoàn toàn JOIN với GDP và Country Codes** vì Global không có số liệu World Bank -> Tiết kiệm 50% thời gian xử lý.
* Gán nhãn 3 giai đoạn đại dịch (`covid_phase`): *Lockdown & Shock (2020)*, *Vaccine & Escapism (2021)*, *Reopening & New Normal (2022)*.

In [ ]:
# Xây dựng Nhánh 2: Dòng thời gian COVID-19
df_covid = build_covid_timeline_dataset(
    charts=charts_clean,
    tracks=tracks_clean,
    artists=artists_clean,
    save=True
)
print(f"Phân bổ 3 giai đoạn đại dịch:\n{df_covid['covid_phase'].value_counts()}")

### Nhánh 3: Dữ liệu Cấp độ Bài hát duy nhất cho Machine Learning (Song-level)
* Gom nhóm theo `track_id` để mỗi bài hát xuất hiện **đúng 1 dòng** (khoảng ~39,202 bài).
* Gán nhãn mục tiêu `is_trending`: 1 nếu bài hát từng lọt Top 50, 0 nếu chỉ dừng ở Top 51–200.
* Giữ lại 9 đặc trưng âm học + `duration_min` + `is_latin` + `is_kpop`.
* **Loại bỏ 100% rủi ro Data Leakage** khi chia tập Train/Test cho bạn Thành Tài.

In [ ]:
# Xây dựng Nhánh 3: Dữ liệu Machine Learning
df_ml = build_ml_dataset(charts=charts_clean, save=True)
df_ml.head(3)

---
## 🧮 BƯỚC 4: KIỂM ĐỊNH CHẤT LƯỢNG CALCULATED FIELDS (FEATURE VALIDATION)
Kiểm tra tính chuẩn xác và phân phối thống kê của các trường tính toán mới được sinh ra.

In [ ]:
# 1. Kiểm tra phân loại cảm xúc Russell (mood_category)
mood_dist = df_socio["mood_category"].value_counts(normalize=True) * 100
print("=== PHÂN BỔ KHÔNG GIAN CẢM XÚC RUSSELL (Nhánh 1) ===")
for mood, pct in mood_dist.items():
    print(f"  • {mood:<20s}: {pct:>5.1f}%")

# 2. Kiểm tra tỷ lệ phát hiện Làn sóng Latin và K-Pop
latin_pct = df_socio["is_latin"].mean() * 100
kpop_pct = df_socio["is_kpop"].mean() * 100
print(f"\n=== TỶ LỆ DÒNG NHẠC PHI TIẾNG ANH ===")
print(f"  • Latin Wave (is_latin = 1) : {latin_pct:.1f}% ({df_socio['is_latin'].sum():,} lượt stream)")
print(f"  • K-Pop Invasion (is_kpop = 1): {kpop_pct:.1f}% ({df_socio['is_kpop'].sum():,} lượt stream)")

---
## 📊 BƯỚC 5: KHÁM PHÁ DỮ LIỆU TĨNH BAN ĐẦU (EDA SANITY CHECKS)
Thực hiện vẽ 3 biểu đồ tĩnh chuẩn mực bằng Matplotlib/Seaborn kiểm tra tính hợp lệ của dữ liệu trước khi chuyển giao cho Dashboard và Model ML.

In [ ]:
# Biểu đồ 1: Ma trận tương quan giữa 9 Audio Features và Lượt nghe (log_streams)
corr_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "log_streams"
]
corr_matrix = df_socio[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Biểu đồ 1: Ma trận Tương quan giữa 9 Đặc trưng Âm học và Lượt nghe (log_streams)", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 2: Diễn biến 3 giai đoạn đại dịch COVID-19 trên bảng Global
covid_summary = df_covid.groupby("covid_phase")[["energy", "valence", "danceability", "acousticness"]].mean().reindex([
    "Lockdown & Shock", "Vaccine & Escapism", "Reopening & New Normal"
])

ax = covid_summary.plot(kind="bar", figsize=(12, 5), colormap="viridis", width=0.7)
plt.title("Biểu đồ 2: So sánh Trung bình Đặc trưng Âm thanh qua 3 Giai đoạn COVID-19 (Global)", fontsize=13, fontweight="bold", pad=15)
plt.ylabel("Điểm số đặc trưng âm thanh (0.0 - 1.0)")
plt.xlabel("Giai đoạn đại dịch")
plt.xticks(rotation=0)
plt.ylim(0, 0.85)
plt.legend(title="Chỉ số", loc="upper right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 3: Phân phối độ dễ nhảy (danceability) và năng lượng (energy) theo nhóm thu nhập World Bank
df_plot = df_socio.dropna(subset=["income_group"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = ["High income", "Upper middle income", "Lower middle income"]
sns.boxplot(data=df_plot, x="income_group", y="danceability", order=order, palette="Set2", ax=axes[0])
axes[0].set_title("Phân phối Danceability theo Nhóm Thu nhập", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Danceability")

sns.boxplot(data=df_plot, x="income_group", y="energy", order=order, palette="Set2", ax=axes[1])
axes[1].set_title("Phân phối Energy theo Nhóm Thu nhập", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Energy")

plt.suptitle("Biểu đồ 3: So sánh Gu Âm nhạc giữa các Phân tầng Thu nhập World Bank", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 💾 BƯỚC 6: TỔNG KẾT & XUẤT CÁC TỆP DỮ LIỆU MỤC TIÊU
Dưới đây là bảng tổng hợp tình trạng và kích thước các tệp dữ liệu đã được xuất tự động vào thư mục `data/processed/` và `data/sample/`:

In [ ]:
output_files = [
    ("Socioeconomic (74 nước 2021-2022)", os.path.join(PROCESSED_DIR, "spotify_socioeconomic_2021_2022.csv")),
    ("COVID Timeline (Global 2020-2022)", os.path.join(PROCESSED_DIR, "spotify_covid_timeline_2020_2022.csv")),
    ("Tracks ML Dataset (Song-level)", os.path.join(PROCESSED_DIR, "tracks_ml_dataset.csv")),
    ("Sample Data (cho Streamlit Dev)", os.path.join(SAMPLE_DIR, "spotify_sample_merged.csv"))
]

summary_data = []
for name, path in output_files:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        df_tmp = pd.read_csv(path, nrows=5)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            n_rows = sum(1 for _ in f) - 1
        summary_data.append({
            "Tên tệp": name,
            "Đường dẫn": os.path.relpath(path, PROJECT_ROOT),
            "Số dòng": f"{n_rows:,}",
            "Số cột": len(df_tmp.columns),
            "Dung lượng": f"{size_mb:.1f} MB",
            "Trạng thái": "✅ Sẵn sàng"
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)
print("\n🎉 HOÀN TẤT TOÀN BỘ CÔNG VIỆC TIỀN XỬ LÝ DỮ LIỆU CỦA QUỐC ANH (MỤC 1 RUBRIC)!")